# 03 · Audit the Relative Wealth Index (H1, H2, H3, H6)

**Before running this on real data:** the `pap-v1` tag must exist on GitHub (implementation guide, step 4).
The tag's date is your proof that the hypotheses came first.

**Time:** about 5 minutes. **Produces:** `results/real/tables/*.csv`, figures, and `docs/results.md`.

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" for you; CI sets MAPAUDIT_NOTEBOOK_MODE=demo to test this notebook
print("Mode:", MODE)

In [ ]:
if MODE == "demo":
    nb.prepare_demo(through="build")
audit = pipeline.audit(MODE)
robust = pipeline.robustness(MODE)
report.report(MODE)
pd.Series(audit, name="value", dtype=object).to_frame()

## The two confirmatory tests

H1 is two-sided, H2 one-sided (coefficient > 0), both at α = 0.05 after Holm's correction, exactly as the
plan says. Everything else in this notebook is an estimate with an uncertainty range, not a test.

In [ ]:
tests = nb.load_table("confirmatory_tests", MODE)
tests[["p_value", "p_holm"]] = tests[["p_value", "p_holm"]].map("{:.1e}".format)  # p-values can be tiny
tests

## Who the audit can see

Share of the rural population each part of the analysis covers. Localities with fewer than three dwellings
have suppressed census data, including Indigenous status, so they get their own row instead of silently
leaving the denominator.

In [ ]:
nb.load_table("evaluation_coverage", MODE).round(3)

## H1 · Who is on the map at all?

In [ ]:
nb.show_figure("rwi_coverage", MODE)
display(nb.load_table("coverage", MODE).round(3))
nb.load_table("coverage_regression", MODE).round(4)

## H2 · Is the rank right, within the same true wealth decile?

Above zero = the map ranks the place as wealthier than the census does. The confirmatory estimate is the
last row of the regression table (all controls).

In [ ]:
nb.show_figure("decile_errors", MODE)
nb.show_figure("audit_specs", MODE)
nb.load_table("audit_regressions", MODE).round(3)

## The user's view: among places the map ranks the same

Same errors, grouped by the map's decile instead of the census's. A gap here means a program that targets by
map rank treats unequally poor places as equals. Groups that differ in average wealth produce some gap here
even for a map with group-blind errors, so read it next to the synthetic no-bias benchmark on the
validation page, not against zero.

In [ ]:
nb.show_figure("map_decile_errors", MODE)
nb.load_table("map_decile_errors", MODE).round(2)

## Robustness (pre-specified)

In [ ]:
robust.round(3)

## H3 · Does the definition of poverty change who counts as poorest?

In [ ]:
nb.show_figure("label_choice", MODE)
display(nb.load_table("label_choice", MODE).round(3))
nb.load_table("label_agreement", MODE).round(3)

## H6 · What would the map's errors cost?

Net change for Indigenous-majority municipalities, then the misallocation share: the fraction of the whole
budget that would have to move to reproduce the census-based allocation.

In [ ]:
nb.show_figure("allocation", MODE)
display(nb.load_table("rwi_targeting", MODE).round(3))
display(nb.load_table("rwi_allocation", MODE).round(3))
nb.load_table("rwi_allocation_errors", MODE).round(3)

In [ ]:
nb.show_figure("map", MODE)

## Your notes (for lab-notebook post 3)

- H1 and H2: what direction, how large, how certain? Report it the same way whichever way it goes.
- How much of the raw gap disappears as controls are added (the specification figure)?
- The user's view: is the gap among places the map ranks the same larger than the synthetic no-bias benchmark?
- Do the robustness checks agree, including the other definitions of poverty? Any regional pattern on the map?
- Anything that forces a change to the plan goes in the deviations table in `docs/pre_analysis_plan.md`.

## Save your work to GitHub

1. Run the next cell (real data only). It runs the checks GitHub will run: formatting, lint, tests and the
   website build. Every row should say `pass`.
2. Commit and push what changed, with a message that says what you did. Nothing from `data/` goes to
   GitHub, and no notebook output may show locality-level rows.
3. On GitHub, the **Actions** tab reruns the checks; every run should go green within about 10 minutes.

In [ ]:
if MODE == "real":
    display(nb.run_checks())